# 04c · Train XLM-RoBERTa emotion classifier (rebalanced English neutral)

This notebook is a copy of notebook 04b with ONE change: the base model is XLM-RoBERTa (FacebookAI/xlm-roberta-base) instead of mBERT.

### Rationale
Rebalancing notebook 04b fixed most of the neutral bias (human English accuracy on HADQAET went from 0.378 to 0.553), but there's still a gap to Chinese (~0.75). XLM-R was trained on more multilingual data than mBERT and is known to transfer better across languages (Conneau et al., 2020), so I want to see if a better multilingual encoder improves it even more.

### Main changes
1. "MODEL_NAME" is XLM-R instead of mBERT
2. It saves to its OWN folder ("models/xlmr_emotion_6class_rebalanced/") so it doesn't overwrite the mBERT-rebalanced model

Code copied from another notebook is marked with where it's from.

#### Outputs
- `models/xlmr_emotion_6class_rebalanced/final/` (the frozen classifier)
- `data/results/xlmr_rebalanced_test_metrics.csv`
- `data/results/xlmr_rebalanced_per_class_metrics.csv`
- `data/results/xlmr_rebalanced_hadqaet_check.csv`

## 0. Put Github repo into Colab

In [ ]:
# Install Git LFS so the large model files download, src: https://huggingface.co/spaces/huggingface/HuggingDiscussions/discussions/5
!apt-get -qq install git-lfs
!git lfs install

# clone repo
import os
if not os.path.exists("/content/W266-slang-emotion-mt"):
    !git clone https://github.com/cierrajagan/W266-slang-emotion-mt.git

%cd /content/W266-slang-emotion-mt

# make sure the real model files download, not the tiny LFS placeholders (has happened before)
!git lfs pull

Selecting previously unselected package git-lfs.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...
Git LFS initialized.
Cloning into 'W266-slang-emotion-mt'...
remote: Enumerating objects: 162, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 162 (delta 61), reused 131 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (162/162), 19.50 MiB | 18.17 MiB/s, done.
Resolving deltas: 100% (61/61), done.


## 1. Install and import dependencies (from notebook 04)

In [2]:
# from notebook 04
%pip install -q -U "transformers>=4.45,<6" datasets accelerate scikit-learn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [3]:
# from notebook 04 (removed imports we don't use here)
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, load_dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed
)


RANDOM_SEED = 266
set_seed(RANDOM_SEED)

#https://huggingface.co/FacebookAI/xlm-roberta-base
MODEL_NAME = "FacebookAI/xlm-roberta-base" #new model
MAX_LENGTH = 128

current_folder = Path.cwd()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder

processed_folder = project_root / "data" / "processed"
results_folder = project_root / "data" / "results"

# NEW: its own model folder so no other classifier gets overwritten
model_folder = project_root / "models" / "xlmr_emotion_6class_rebalanced"
final_model_folder = model_folder / "final"

results_folder.mkdir(parents=True, exist_ok=True)
model_folder.mkdir(parents=True, exist_ok=True)

usual_splits_path = processed_folder / "usual_splits.csv"
post_edits_path = processed_folder / "hadqaet_post_edits.csv"   # NEW: for the HADQAET check at the end

print("Project root:", project_root)
print("Input:", usual_splits_path, "| exists:", usual_splits_path.exists())
print("HADQAET check file:", post_edits_path, "| exists:", post_edits_path.exists())
print("Base model:", MODEL_NAME)
print("Model output:", model_folder)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Project root: /content/W266-slang-emotion-mt
Input: /content/W266-slang-emotion-mt/data/processed/usual_splits.csv | exists: True
HADQAET check file: /content/W266-slang-emotion-mt/data/processed/hadqaet_post_edits.csv | exists: True
Base model: FacebookAI/xlm-roberta-base
Model output: /content/W266-slang-emotion-mt/models/xlmr_emotion_6class_rebalanced
CUDA available: True
GPU: Tesla T4


## 2. Load SMP usual data (from notebook 04)

SMP rows overlapping HADQAET are not used for training (`mbert_train_eligible == True` only), so HADQAET can be used as a check later.

In [4]:
# from notebook 04
smp = pd.read_csv(usual_splits_path, encoding="utf-8-sig")

required_columns = {"post_id", "split", "text", "label", "in_hadqaet"}
missing = required_columns - set(smp.columns)

if missing:
  raise ValueError(f"Missing columns: {missing}")

print("Rows:", len(smp))
print("\nSplit counts:")
print(smp["split"].value_counts())
print("\nHADQAET-overlapping rows:", int(smp["in_hadqaet"].astype(bool).sum()))
print("Train-eligible rows:", int(smp["mbert_train_eligible"].astype(bool).sum()))

Rows: 34766

Split counts:
split
train    27864
eval      3457
test      3445
Name: count, dtype: int64

HADQAET-overlapping rows: 5977
Train-eligible rows: 21887


## 3. Create six label classes (from notebook 04)

In [5]:
# from notebook 04
LABELS = ["joy", "anger", "sadness", "fear", "surprise", "neutral"]

label_to_ids = {label: idx for idx, label in enumerate(LABELS)}
ids_to_label = {idx: label for label, idx in label_to_ids.items()}

# Map smp labels to shared labels
SMP_TO_SHARED = {
    "happy": "joy",
    "angry": "anger",
    "sad": "sadness",
    "fear": "fear",
    "surprise": "surprise",
    "neutral": "neutral"
}

smp["shared_label"] = smp["label"].map(SMP_TO_SHARED)

# Validate no unmapped
unmapped_smp = smp.loc[smp["shared_label"].isna(), "label"].unique().tolist()
if unmapped_smp:
    raise ValueError(f"Unmapped SMP labels found: {unmapped_smp}")

print("Shared labels:", LABELS)
print("Label ids:", label_to_ids)

Shared labels: ['joy', 'anger', 'sadness', 'fear', 'surprise', 'neutral']
Label ids: {'joy': 0, 'anger': 1, 'sadness': 2, 'fear': 3, 'surprise': 4, 'neutral': 5}


## 4. Prepare Chinese SMP train/validation/test sets (from notebook 04)

In [6]:
# from notebook 04
smp["normalized_text"] = smp["text"].astype(str).str.strip()

def deduplicate_text(df):
    return (
        df
        .drop_duplicates(subset=["normalized_text"], keep="first")
        .copy()
        .reset_index(drop=True)
    )

# Only take train eligible from the train split
train_raw = deduplicate_text(
    smp[smp["mbert_train_eligible"].astype(bool)]
)
val_raw = deduplicate_text(
    smp[smp["split"] == "eval"]
)
test_raw = deduplicate_text(
    smp[smp["split"] == "test"]
)

assert not train_raw["in_hadqaet"].astype(bool).any()

train_texts = set(train_raw["normalized_text"])
val_texts = set(val_raw["normalized_text"])
test_texts = set(test_raw["normalized_text"])

assert train_texts.isdisjoint(val_texts)
assert train_texts.isdisjoint(test_texts)
assert val_texts.isdisjoint(test_texts)

print("Chinese train:", len(train_raw))
print("Chinese validation:", len(val_raw))
print("Chinese test:", len(test_raw))
print("\nChinese train label distribution:")
print(train_raw["shared_label"].value_counts())

Chinese train: 21375
Chinese validation: 3360
Chinese test: 3360

Chinese train label distribution:
shared_label
neutral     5702
anger       5604
joy         3997
sadness     3720
surprise    1491
fear         861
Name: count, dtype: int64


## 5. Load English GoEmotions data (from notebook 04)

https://huggingface.co/datasets/google-research-datasets/go_emotions

In [7]:
# from notebook 04
go = load_dataset(
    "google-research-datasets/go_emotions",
    "simplified",
)

go_label_names = go["train"].features["labels"].feature.names

print("GoEmotions label count:", len(go_label_names))
print(go_label_names)

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.77MB            

simplified/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

simplified/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  350kB            

simplified/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

simplified/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  347kB            

simplified/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

GoEmotions label count: 28
['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [8]:
# from notebook 04
def filter_goemotions(hf_split):
    rows = []
    for example in hf_split:
        label_names = [
            go_label_names[label_id]
            for label_id in example["labels"]
        ]

        if len(label_names) == 1 and label_names[0] in LABELS:
            rows.append({
                "text": example["text"],
                "shared_label": label_names[0],
            })

    return pd.DataFrame(rows)

en_train_raw = filter_goemotions(go["train"])
en_val_raw = filter_goemotions(go["validation"])
en_test_raw = filter_goemotions(go["test"])


print("English train:", len(en_train_raw))
print("English validation:", len(en_val_raw))
print("English test:", len(en_test_raw))
print("\nEnglish train label distribution (BEFORE rebalancing):")
print(en_train_raw["shared_label"].value_counts())

English train: 16668
English validation: 2044
English test: 2084

English train label distribution (BEFORE rebalancing):
shared_label
neutral     12823
anger        1025
joy           853
sadness       817
surprise      720
fear          430
Name: count, dtype: int64


## 6. Rebalance English neutral (from notebook 04b)

Same rebalancing as notebook 04b: downsample GoEmotions neutral in train + validation to match the Chinese neutral share. Test stays at its natural distribution.

In [9]:
# from notebook 04b
# The share of neutral in the Chinese training data is the target for English
chinese_neutral_share = (train_raw["shared_label"] == "neutral").mean()
print(f"Chinese train neutral share: {chinese_neutral_share:.3f}")


def downsample_neutral(df, target_share, seed):
    neutral_df = df[df["shared_label"] == "neutral"]
    other_df = df[df["shared_label"] != "neutral"]

    # How many neutral examples to keep so that neutral = target_share of the result:
    # n_neutral / (n_neutral + n_other) = target_share
    # -> n_neutral = n_other * target_share / (1 - target_share)
    n_keep = int(round(len(other_df) * target_share / (1 - target_share)))
    n_keep = min(n_keep, len(neutral_df))   # can't keep more than we have

    neutral_kept = neutral_df.sample(n=n_keep, random_state=seed)

    # Put them back together and shuffle (random_state so it's reproducible)
    return (
        pd.concat([other_df, neutral_kept], ignore_index=True)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )


en_train_balanced = downsample_neutral(en_train_raw, chinese_neutral_share, RANDOM_SEED)
en_val_balanced = downsample_neutral(en_val_raw, chinese_neutral_share, RANDOM_SEED)

for name, before, after in [
    ("train", en_train_raw, en_train_balanced),
    ("validation", en_val_raw, en_val_balanced),
]:
    print(f"\nEnglish {name}: {len(before)} -> {len(after)} rows")
    print(f"  neutral share: {(before['shared_label'] == 'neutral').mean():.3f} -> "
          f"{(after['shared_label'] == 'neutral').mean():.3f}")

print("\nEnglish train label distribution (AFTER rebalancing):")
print(en_train_balanced["shared_label"].value_counts())

Chinese train neutral share: 0.267

English train: 16668 -> 5244 rows
  neutral share: 0.769 -> 0.267

English validation: 2044 -> 616 rows
  neutral share: 0.779 -> 0.266

English train label distribution (AFTER rebalancing):
shared_label
neutral     1399
anger       1025
joy          853
sadness      817
surprise     720
fear         430
Name: count, dtype: int64


## 7. Build multilingual train/validation/test sets (from notebook 04)

Only change: English train/validation use the rebalanced versions.

In [10]:
# from notebook 04
def to_classifier_frame(df, language, source):
  output_df = df[["text", "shared_label"]].copy()
  output_df["labels"] = output_df["shared_label"].map(label_to_ids)
  output_df["language"] = language
  output_df["source"] = source

  if output_df["labels"].isna().any():
        raise ValueError(f"Unmapped labels found in {source}")

  return output_df[["text", "shared_label", "labels", "language", "source"]]

In [11]:
# from notebook 04
cn_train_df = to_classifier_frame(train_raw, "Chinese", "SMP")
cn_val_df = to_classifier_frame(val_raw, "Chinese", "SMP")
cn_test_df = to_classifier_frame(test_raw, "Chinese", "SMP")

# CHANGED: rebalanced English train/validation (test keeps its natural distribution)
en_train_df = to_classifier_frame(en_train_balanced, "English", "GoEmotions")
en_val_df = to_classifier_frame(en_val_balanced, "English", "GoEmotions")
en_test_df = to_classifier_frame(en_test_raw, "English", "GoEmotions")

# Combine Chinese and English, shuffle with random_state so it's reproducible
train_df = pd.concat([cn_train_df, en_train_df], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

val_df = pd.concat([cn_val_df, en_val_df], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

test_df = pd.concat([cn_test_df, en_test_df], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print("Total train:", len(train_df))
print("Total validation:", len(val_df))
print("Total test:", len(test_df))

print("\nTraining counts by language and emotion:")
display(pd.crosstab(train_df["shared_label"], train_df["language"]))

print("\nValidation counts by language and emotion:")
display(pd.crosstab(val_df["shared_label"], val_df["language"]))

Total train: 26619
Total validation: 3976
Total test: 5444

Training counts by language and emotion:


language,Chinese,English
shared_label,,
anger,5604,1025
fear,861,430
joy,3997,853
neutral,5702,1399
sadness,3720,817
surprise,1491,720



Validation counts by language and emotion:


language,Chinese,English
shared_label,,
anger,956,109
fear,145,58
joy,674,106
neutral,712,164
sadness,619,84
surprise,254,95


## 8. Define and fine-tune XLM-R (from notebook 04)

Same hyperparameters and seed as notebook 04: learning rate 2e-5, 3 epochs, weight decay 0.01, batch size 16/32, best checkpoint picked by validation macro-F1.

In [12]:
# from notebook 04
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

# Keep only text and labels
train_dataset = Dataset.from_pandas(train_df[["text", "labels"]])
val_dataset = Dataset.from_pandas(val_df[["text", "labels"]])

# tokenize train and val datasets
tokenized_train_ds = train_dataset.map(tokenize_function, batched=True)
tokenized_val_ds = val_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# AutoModelForSequenceClassification adds the six-class head on top of the base model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ids_to_label,
    label2id=label_to_ids,
)

print("Base model:", MODEL_NAME)
print("Number of output classes:", model.config.num_labels)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/26619 [00:00<?, ? examples/s]

Map:   0%|          | 0/3976 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Base model: FacebookAI/xlm-roberta-base
Number of output classes: 6


In [13]:
# from notebook 04
def compute_metrics(eval_pred):
    logits, true_labels = eval_pred
    predicted_labels = np.argmax(logits, axis=-1)

    return {"accuracy": accuracy_score(true_labels, predicted_labels),
            "macro_f1": f1_score(true_labels, predicted_labels,
                                 labels=list(range(len(LABELS))),
                                 average="macro", zero_division=0)
            }

In [14]:
# from notebook 04
use_bf16 = (
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)

training_args = TrainingArguments(
    output_dir=str(model_folder / "checkpoints"),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay= 0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    logging_strategy="steps",
    logging_steps=100,
    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,
    report_to="none",
    seed=RANDOM_SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_ds,
    eval_dataset=tokenized_val_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation macro-F1:", trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.718593,0.767790,0.740191,0.719459
2,0.549340,0.645232,0.778169,0.758621
3,0.409762,0.695825,0.782445,0.765186


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best checkpoint: /content/W266-slang-emotion-mt/models/xlmr_emotion_6class_rebalanced/checkpoints/checkpoint-4992
Best validation macro-F1: 0.7651859519904934


### Freeze the best model (from notebook 04b)

"load_best_model_at_end=True" means "trainer.model" is already the best checkpoint. I save the model + tokenizer to one fixed "final/" folder so notebooks 05 and 05b don't depend on a checkpoint number.

In [15]:
# from notebook 04b
trainer.save_model(str(final_model_folder))
tokenizer.save_pretrained(str(final_model_folder))

print("Saved frozen classifier to:", final_model_folder)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved frozen classifier to: /content/W266-slang-emotion-mt/models/xlmr_emotion_6class_rebalanced/final


## 9. Validate on test sets (from notebook 04)

Chinese SMP test, English GoEmotions test (natural distribution, so comparable with notebook 04) and combined

Added a "pct_pred_neutral" column so we can see the neutral results directly

In [16]:
# from notebook 04
def predict_output_df(df):
  dataset = Dataset.from_pandas(df[["text", "labels"]], preserve_index=True)
  tokenized_ds = dataset.map(tokenize_function, batched=True)
  output = trainer.predict(tokenized_ds)
  y_true = df["labels"].to_numpy()

  # the shape is [n, 6], argmax(axis=1) to get the column with the highest probability for each sample
  y_pred = output.predictions.argmax(axis=1)

  return y_true, y_pred

In [17]:
# from notebook 04, with a model name from this notebook and a NEW % predicted neutral column
MODEL_LABEL = "Finetuned-XLMR-rebalanced"

def summarize_test_set(df, dataset_name):
  y_true, y_pred = predict_output_df(df)

  summary = {
      "model": MODEL_LABEL,
      "dataset": dataset_name,
      "n":len(df),
      "accuracy": accuracy_score(y_true, y_pred),
      "macro_f1": f1_score(y_true, y_pred, labels=list(range(len(LABELS))), average="macro", zero_division=0),
      # NEW: how often the model predicts neutral (the bias we're trying to fix)
      "pct_pred_neutral": float((y_pred == label_to_ids["neutral"]).mean() * 100),
      }
  return summary

def summarize_per_class_stats(df, dataset_name):
    y_true, y_pred = predict_output_df(df)
    precision, recall, per_class_f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(len(LABELS))), average=None, zero_division=0)
    per_class_df = pd.DataFrame({
        "model": MODEL_LABEL,
        "dataset": dataset_name,
        "label": LABELS,
        "precision": precision,
        "recall": recall,
        "f1": per_class_f1,
        "support": support,
    })

    return per_class_df

In [18]:
# from notebook 04
print("SMP Chinese Test")
cn_summary = summarize_test_set(cn_test_df, "SMP Chinese Test")
cn_per_class_summary = summarize_per_class_stats(cn_test_df, "SMP Chinese Test")

print("\nGoEmotions English Test")
en_summary = summarize_test_set(en_test_df, "GoEmotions English Test")
en_per_class_summary = summarize_per_class_stats(en_test_df, "GoEmotions English Test")

print("\nCombined Multilingual Test")
combined_summary = summarize_test_set(test_df, "Combined Multilingual Test")
combined_per_class_summary = summarize_per_class_stats(test_df, "Combined Multilingual Test")

test_metrics = pd.DataFrame([
    cn_summary,
    en_summary,
    combined_summary,
])

display(test_metrics)

SMP Chinese Test


Map:   0%|          | 0/3360 [00:00<?, ? examples/s]

Map:   0%|          | 0/3360 [00:00<?, ? examples/s]


GoEmotions English Test


Map:   0%|          | 0/2084 [00:00<?, ? examples/s]

Map:   0%|          | 0/2084 [00:00<?, ? examples/s]


Combined Multilingual Test


Map:   0%|          | 0/5444 [00:00<?, ? examples/s]

Map:   0%|          | 0/5444 [00:00<?, ? examples/s]

,model,dataset,n,accuracy,macro_f1,pct_pred_neutral
0,Finetuned-XLMR-rebalanced,SMP Chinese Test,3360,0.781548,0.748664,21.428571
1,Finetuned-XLMR-rebalanced,GoEmotions English Test,2084,0.721689,0.612914,55.086372
2,Finetuned-XLMR-rebalanced,Combined Multilingual Test,5444,0.758633,0.717884,34.313005


In [19]:
# from notebook 04
per_class_metrics = pd.concat(
    [cn_per_class_summary, en_per_class_summary, combined_per_class_summary],
    ignore_index=True,
)

display(per_class_metrics)

# Save to files (file names from this notebook so nothing gets overwritten)
test_metrics_path = results_folder / "xlmr_rebalanced_test_metrics.csv"
per_class_path = results_folder / "xlmr_rebalanced_per_class_metrics.csv"

test_metrics.to_csv(test_metrics_path, index=False)
per_class_metrics.to_csv(per_class_path, index=False)

print("Saved:", test_metrics_path)
print("Saved:", per_class_path)

,model,dataset,label,precision,recall,f1,support
0,Finetuned-XLMR-rebalanced,SMP Chinese Test,joy,0.834131,0.775964,0.803997,674
1,Finetuned-XLMR-rebalanced,SMP Chinese Test,anger,0.836364,0.817992,0.827076,956
2,Finetuned-XLMR-rebalanced,SMP Chinese Test,sadness,0.673879,0.704362,0.688784,619
3,Finetuned-XLMR-rebalanced,SMP Chinese Test,fear,0.668790,0.724138,0.695364,145
4,Finetuned-XLMR-rebalanced,SMP Chinese Test,surprise,0.591241,0.637795,0.613636,254
5,Finetuned-XLMR-rebalanced,SMP Chinese Test,neutral,0.858333,0.867978,0.863128,712
6,Finetuned-XLMR-rebalanced,GoEmotions English Test,joy,0.468208,0.870968,0.609023,93
7,Finetuned-XLMR-rebalanced,GoEmotions English Test,anger,0.329114,0.793893,0.465324,131
8,Finetuned-XLMR-rebalanced,GoEmotions English Test,sadness,0.421875,0.794118,0.551020,102
9,Finetuned-XLMR-rebalanced,GoEmotions English Test,fear,0.591398,0.846154,0.696203,65


Saved: /content/W266-slang-emotion-mt/data/results/xlmr_rebalanced_test_metrics.csv
Saved: /content/W266-slang-emotion-mt/data/results/xlmr_rebalanced_per_class_metrics.csv


## 10. HADQAET check (from notebook 05b)

Exact same check as notebook 05b, using the same helper functions, so the numbers are directly comparable: classify HADQAET's Chinese originals, Google MT English and human-quality English. HADQAET posts were never used for training and have no neutral gold labels, so every neutral prediction is wrong.

For comparison:

| classifier | text version | accuracy | macro-F1 | % predicted neutral |
|---|---|---|---|---|
| mBERT original (nb 04) | Chinese original | 0.753 | 0.734 | 4.3 |
| mBERT original (nb 04) | Google MT English | 0.335 | 0.467 | 55.3 |
| mBERT original (nb 04) | Human-quality English | 0.378 | 0.499 | 49.9 |
| mBERT rebalanced (nb 04b) | Chinese original | 0.749 | 0.732 | 4.7 |
| mBERT rebalanced (nb 04b) | Google MT English | 0.507 | 0.549 | 24.0 |
| mBERT rebalanced (nb 04b) | Human-quality English | 0.553 | 0.578 | 19.3 |

What I'm hoping to see: % predicted neutral on English drops (towards the Chinese ~4%), English accuracy goes up, and Chinese accuracy stays about the same.

In [20]:
# The helper functions from notebook 05b expect these names:
# model, tokenizer, device, LABELS, post_edits_path
# trainer.model is already the best checkpoint (load_best_model_at_end=True)
model = trainer.model
model.eval()
device = model.device
print("Running on:", device)

Running on: cuda:0


In [21]:
# from notebook 05b
# Helper function: predict an emotion for each text in a list
# texts = a list of strings
# batch_size = how many texts to send to the model at once
# Returns a list of predicted labels, e.g. ["anger", "neutral", ...]

def predict_emotions(texts, batch_size=32):
    predictions = []

    # Go through the texts one batch at a time
    for batch_start in range(0, len(texts), batch_size):
        batch_texts = texts[batch_start:batch_start + batch_size]

        # Turn the texts into tokens (same settings as notebooks 04 and 05: max 128 tokens)
        inputs = tokenizer(
            batch_texts,
            truncation=True,
            max_length=128,
            padding=True,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad(): # torch.no_grad() = we're only predicting, so don't track gradients (faster), src: https://stackoverflow.com/questions/68579174/when-to-use-torch-no-grad-is-safe-in-forward-propagation-why-does-it-hurt-my
            logits = model(**inputs).logits

        # The highest score in each row is the predicted class
        predicted_ids = logits.argmax(dim=1).tolist()
        for predicted_id in predicted_ids:
            predictions.append(LABELS[predicted_id])

    return predictions

In [22]:
# from notebook 05b
hadqaet = pd.read_csv(post_edits_path)
print("HADQAET rows:", len(hadqaet))

# Clean the gold labels (strip spaces, lowercase)
gold_labels = []
for label in hadqaet["emotion_labels"]:
    gold_labels.append(str(label).strip().lower())
hadqaet["gold"] = gold_labels

# make sure every gold label is one of our 6 labels
unknown = set(hadqaet["gold"]) - set(LABELS)
print("Labels not in our set (should be empty):", unknown)

# Build the human-quality English column
human_english = []
has_post_edit = []
for index, row in hadqaet.iterrows():
    post_edit = row["post-edit"]
    if pd.isna(post_edit) or str(post_edit).strip() == "":
        # If no error was annotated, then Google's translation was correct
        human_english.append(str(row["MT"]))
        has_post_edit.append(False)
    else:
        human_english.append(str(post_edit))
        has_post_edit.append(True)

hadqaet["human_english"] = human_english
hadqaet["has_post_edit"] = has_post_edit

print("Num post-edit:", sum(has_post_edit))
print("Num correct Google MT:", len(hadqaet) - sum(has_post_edit))
print(hadqaet["gold"].value_counts())

HADQAET rows: 4038
Labels not in our set (should be empty): set()
Num post-edit: 2952
Num correct Google MT: 1086
gold
anger       1429
joy         1187
sadness      887
surprise     282
fear         253
Name: count, dtype: int64


In [23]:
# from notebook 05b
# Helper function: accuracy, macro-F1 and % predicted neutral

def score(gold, predicted):
    accuracy = accuracy_score(gold, predicted)

    # Macro-F1 over the 5 emotions that actually occur in HADQAET (no neutral gold labels)
    macro_f1 = f1_score(
        gold, predicted,
        labels=["joy", "anger", "sadness", "fear", "surprise"],
        average="macro",
        zero_division=0,
    )

    # count neutral predictions
    neutral_count = 0
    for label in predicted:
        if label == "neutral":
            neutral_count = neutral_count + 1
    percent_neutral = neutral_count / len(predicted) * 100

    return accuracy, macro_f1, percent_neutral

In [24]:
# from notebook 05b (all posts only, with the model name added)
versions = {
    "Chinese original": "source",
    "Google MT English": "MT",
    "Human-quality English": "human_english",
}

hadqaet_rows = []
for version_name in versions:
    text_column = versions[version_name]
    print("Classifying", version_name)
    predicted = predict_emotions(hadqaet[text_column].astype(str).tolist())

    accuracy, macro_f1, percent_neutral = score(hadqaet["gold"], predicted)
    hadqaet_rows.append({
        "model": MODEL_LABEL,
        "text version": version_name,
        "n": len(hadqaet),
        "accuracy": round(accuracy, 3),
        "macro_f1": round(macro_f1, 3),
        "% predicted neutral": round(percent_neutral, 1),
    })

hadqaet_check = pd.DataFrame(hadqaet_rows)
display(hadqaet_check)

hadqaet_check_path = results_folder / "xlmr_rebalanced_hadqaet_check.csv"
hadqaet_check.to_csv(hadqaet_check_path, index=False)
print("Saved:", hadqaet_check_path)

Classifying Chinese original
Classifying Google MT English
Classifying Human-quality English


,model,text version,n,accuracy,macro_f1,% predicted neutral
0,Finetuned-XLMR-rebalanced,Chinese original,4038,0.789,0.767,3.4
1,Finetuned-XLMR-rebalanced,Google MT English,4038,0.556,0.605,23.3
2,Finetuned-XLMR-rebalanced,Human-quality English,4038,0.600,0.628,18.7


Saved: /content/W266-slang-emotion-mt/data/results/xlmr_rebalanced_hadqaet_check.csv


## 11. Saving the model

The `final/` folder is ~700 MB (mBERT) to ~1 GB (XLM-R). GitHub's free Git LFS quota is small, so I'm saving it to Google Drive instead of pushing it to the repo. Don't commit the `checkpoints/` folder at all.

To use this classifier in notebook 05 or 05b, point the model path at the `final/` folder.

In [26]:
# Copy the frozen model to Google Drive
from google.colab import drive
drive.mount("/content/drive")

drive_folder = "/content/drive/MyDrive/ISchool/MIDS/266/models/" + model_folder.name
!mkdir -p "{drive_folder}"
!cp -r "{final_model_folder}" "{drive_folder}/"
print("Copied to:", drive_folder)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied to: /content/drive/MyDrive/ISchool/MIDS/266/models/xlmr_emotion_6class_rebalanced
